# PROJET FIN D'ÉTUDES - INGÉNIEUR IA
# PHASE 3 : MODÉLISATION

Blocs : 3A (Prophet/SARIMA - volumes), 3B (Ridge/ElasticNet/XGBoost - charge par ETP), 3C (RandomForest/DecisionTree - classification retard), 3D (Isolation Forest + SHAP - anomalies)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from datetime import datetime, date, timedelta
warnings.filterwarnings('ignore')

## PROJET FIN D'ÉTUDES - INGÉNIEUR IA
PHASE 3 : MODÉLISATION
BLOC 3A — PRÉVISION DES VOLUMES (Prophet vs SARIMA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from datetime import datetime, date, timedelta
warnings.filterwarnings('ignore')

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

print("=" * 60)
print("PHASE 3 - MODÉLISATION")
print("BLOC 3A - PRÉVISION DES VOLUMES (Prophet vs SARIMA)")
print("=" * 60)

## DÉTECTION AUTOMATIQUE DE LA DATE DU JOUR

In [ ]:
TODAY    = datetime.today().date()
TODAY_DT = pd.Timestamp(TODAY)
NOM_JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
NOM_MOIS  = ["", "Janvier", "Février", "Mars", "Avril", "Mai", "Juin",
              "Juillet", "Août", "Septembre", "Octobre", "Novembre", "Décembre"]

print(f"\n📅 Date du jour : {NOM_JOURS[TODAY.weekday()]} {TODAY.day} {NOM_MOIS[TODAY.month]} {TODAY.year}")

## CHARGEMENT DES DONNÉES

In [ ]:
print("\n📂 Chargement de dataset_journalier.csv...")

df_jour = pd.read_csv("dataset_journalier.csv", parse_dates=["date"])
df_jour = df_jour.sort_values("date").reset_index(drop=True)

DERNIERE_DATE_CONNUE = df_jour["date"].max().date()
ECART_JOURS = (TODAY - DERNIERE_DATE_CONNUE).days

print(f"  ✅ {len(df_jour)} jours | {df_jour['date'].min().date()} → {DERNIERE_DATE_CONNUE}")
print(f"  → Écart données / aujourd'hui : {ECART_JOURS} jours")

TARGET = "volume_entrant_jour"

## SPLIT TEMPOREL 70 / 15 / 15

In [ ]:
n       = len(df_jour)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

df_train = df_jour.iloc[:n_train].copy()
df_val   = df_jour.iloc[n_train:n_train + n_val].copy()
df_test  = df_jour.iloc[n_train + n_val:].copy()

print(f"\n  Split chronologique :")
print(f"    Train      : {len(df_train)} jours ({df_train['date'].min().date()} → {df_train['date'].max().date()})")
print(f"    Validation : {len(df_val)} jours ({df_val['date'].min().date()} → {df_val['date'].max().date()})")
print(f"    Test       : {len(df_test)} jours ({df_test['date'].min().date()} → {df_test['date'].max().date()})")


def metriques(y_true, y_pred, label, dates=None):
    """Calcule MAE, RMSE, MAPE et les affiche.
    Si `dates` est fourni, calcule aussi un MAPE restreint aux jours ouvrés
    (lun-ven), car le MAPE global est instable quand les valeurs week-end
    sont très faibles (un petit écart absolu donne un % énorme)."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true))) * 100
    print(f"\n  📊 Métriques [{label}]")
    print(f"    MAE  : {mae:.1f}  dossiers")
    print(f"    RMSE : {rmse:.1f} dossiers")
    print(f"    MAPE (tous jours) : {mape:.1f} %")
    if dates is not None:
        dow = pd.to_datetime(dates).dt.dayofweek
        mask = dow < 5
        if mask.sum() > 0:
            yt, yp = np.asarray(y_true)[mask], np.asarray(y_pred)[mask]
            mape_ouvre = np.mean(np.abs((yt - yp) / np.where(yt == 0, np.nan, yt))) * 100
            print(f"    MAPE (jours ouvrés lun-ven) : {mape_ouvre:.1f} %")
    return mae, rmse, mape

## MODÈLE 1 — PROPHET

In [ ]:
print("\n" + "=" * 60)
print("MODÈLE 1 — PROPHET")
print("=" * 60)

# Prophet exige les colonnes 'ds' (date) et 'y' (valeur cible)
def ajouter_regressors(df_in):
    """Ajoute les colonnes de régresseurs externes (fin de trimestre, fin de mois,
    creux estival) calculées à partir de la date, pour aider Prophet à capter
    la saisonnalité forte injectée dans les données (cf. générateur)."""
    out = df_in.copy()
    d = pd.to_datetime(out["ds"])
    out["est_fin_trimestre_reg"] = d.dt.month.isin([3, 6, 9, 12]).astype(int)
    out["est_fin_mois_reg"]      = (d.dt.day >= 25).astype(int)
    out["est_aout_reg"]          = (d.dt.month == 8).astype(int)
    return out


df_prophet_train = df_train[["date", TARGET]].rename(columns={"date": "ds", TARGET: "y"})
df_prophet_train = ajouter_regressors(df_prophet_train)

model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode="additive",
    changepoint_prior_scale=0.05,
)
model_prophet.add_regressor("est_fin_trimestre_reg")
model_prophet.add_regressor("est_fin_mois_reg")
model_prophet.add_regressor("est_aout_reg")

print("\n  Entraînement Prophet sur le train set...")
model_prophet.fit(df_prophet_train)

# Prévisions sur validation + test (mêmes dates)
df_future_val_test = pd.concat([df_val, df_test])[["date"]].rename(columns={"date": "ds"})
df_future_val_test = ajouter_regressors(df_future_val_test)
forecast = model_prophet.predict(df_future_val_test)

pred_val_prophet  = forecast["yhat"].values[:len(df_val)]
pred_test_prophet = forecast["yhat"].values[len(df_val):]

print("\n  Évaluation sur Validation et Test :")
metriques(df_val[TARGET].values,  pred_val_prophet,  "Prophet - Validation", df_val["date"])
mae_test_p, rmse_test_p, mape_test_p = metriques(
    df_test[TARGET].values, pred_test_prophet, "Prophet - Test", df_test["date"]
)

## MODÈLE 2 — SARIMA

In [ ]:
print("\n" + "=" * 60)
print("MODÈLE 2 — SARIMA")
print("=" * 60)

# SARIMA a besoin d'une série indexée par date, fréquence journalière
serie_train = df_train.set_index("date")[TARGET].asfreq("D").interpolate()

print("\n  Entraînement SARIMA (order=(1,1,1), seasonal_order=(1,1,1,7))...")
print("  → saisonnalité hebdomadaire (période 7) : cohérent avec l'activité ouvrée Lun-Ven")

model_sarima = SARIMAX(
    serie_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fit_sarima = model_sarima.fit(disp=False)

n_val_test = len(df_val) + len(df_test)
forecast_sarima = fit_sarima.get_forecast(steps=n_val_test)
pred_sarima_all = forecast_sarima.predicted_mean.values

pred_val_sarima  = pred_sarima_all[:len(df_val)]
pred_test_sarima = pred_sarima_all[len(df_val):]

print("\n  Évaluation sur Validation et Test :")
metriques(df_val[TARGET].values,  pred_val_sarima,  "SARIMA - Validation", df_val["date"])
mae_test_s, rmse_test_s, mape_test_s = metriques(
    df_test[TARGET].values, pred_test_sarima, "SARIMA - Test", df_test["date"]
)

## COMPARAISON ET SÉLECTION DU MODÈLE

In [ ]:
print("\n" + "=" * 60)
print("COMPARAISON PROPHET vs SARIMA (jeu de test)")
print("=" * 60)

comparaison = pd.DataFrame({
    "Modèle": ["Prophet", "SARIMA"],
    "MAE":  [mae_test_p, mae_test_s],
    "RMSE": [rmse_test_p, rmse_test_s],
    "MAPE (%)": [mape_test_p, mape_test_s],
})
print("\n" + comparaison.to_string(index=False))

ecart_relatif = abs(mae_test_p - mae_test_s) / mae_test_s * 100
if ecart_relatif < 2:
    print(f"\n  ⚖️  Écart MAE Prophet/SARIMA < 2% ({ecart_relatif:.1f}%) — performances quasi équivalentes.")
    print("  ✅ Modèle retenu pour la production : PROPHET")
    print("     (intervalles de confiance yhat_lower/yhat_upper utiles pour le dashboard,")
    print("      SARIMA conservé comme comparatif de validation méthodologique)")
    modele_gagnant = "Prophet (ex-aequo avec SARIMA)"
elif mae_test_p < mae_test_s:
    modele_gagnant = "Prophet"
    print("\n  ✅ Modèle retenu pour la production : PROPHET (meilleur MAE)")
else:
    modele_gagnant = "SARIMA"
    print("\n  ✅ Modèle retenu pour la production : SARIMA (meilleur MAE)")

## GRAPHIQUE — PRÉVISION VS RÉEL (TEST)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_test["date"], df_test[TARGET], label="Réel", color="#2c7bb6", linewidth=1.6)
ax.plot(df_test["date"], pred_test_prophet, label="Prophet", color="#d7191c",
        linewidth=1.4, linestyle="--")
ax.plot(df_test["date"], pred_test_sarima, label="SARIMA", color="#f0883e",
        linewidth=1.4, linestyle=":")
ax.set_title("Prévision du volume entrant — Prophet vs SARIMA (jeu de test)",
              fontsize=12, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Volume entrant (dossiers/jour)")
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig("volumes_prophet_vs_sarima.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  ✅ Graphique sauvegardé : volumes_prophet_vs_sarima.png")

## PRÉVISIONS FUTURES ANCRÉES SUR AUJOURD'HUI (J+1 à J+30)

In [ ]:
print(f"\n  Génération des prévisions futures à partir d'aujourd'hui ({TODAY})...")

# On ré-entraîne sur TOUT l'historique disponible pour la prévision finale en prod
df_prophet_full = df_jour[["date", TARGET]].rename(columns={"date": "ds", TARGET: "y"})
df_prophet_full = ajouter_regressors(df_prophet_full)

model_prophet_full = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode="additive",
    changepoint_prior_scale=0.05,
)
model_prophet_full.add_regressor("est_fin_trimestre_reg")
model_prophet_full.add_regressor("est_fin_mois_reg")
model_prophet_full.add_regressor("est_aout_reg")
model_prophet_full.fit(df_prophet_full)

# Les données n'existent que pour les jours ouvrés (banque fermée le week-end) :
# on génère donc les prévisions futures uniquement sur les jours ouvrés (Lun-Ven),
# pour rester cohérent avec l'historique d'entraînement.
dates_candidates = pd.date_range(start=TODAY_DT + timedelta(days=1), periods=45, freq="D")
dates_futures = dates_candidates[dates_candidates.dayofweek < 5][:30]
df_futur = pd.DataFrame({"ds": dates_futures})
df_futur = ajouter_regressors(df_futur)
forecast_futur = model_prophet_full.predict(df_futur)

df_prevision_volumes = pd.DataFrame({
    "date": dates_futures,
    "horizon_j": range(1, len(dates_futures) + 1),
    "jour_semaine": [NOM_JOURS[d.weekday()] for d in dates_futures],
    "volume_prevu": forecast_futur["yhat"].round(0).astype(int).values,
    "volume_prevu_min": forecast_futur["yhat_lower"].round(0).astype(int).values,
    "volume_prevu_max": forecast_futur["yhat_upper"].round(0).astype(int).values,
})

df_prevision_volumes.to_csv("previsions_volumes_j30.csv", index=False, encoding="utf-8")
print(f"  ✅ previsions_volumes_j30.csv exporté")
print(f"\n  Prévisions J+1 à J+7 :")
print(df_prevision_volumes[df_prevision_volumes["horizon_j"] <= 7]
      [["date", "jour_semaine", "volume_prevu"]].to_string(index=False))

## RAPPORT FINAL

In [ ]:
print("\n" + "=" * 60)
print("RAPPORT FINAL - BLOC 3A")
print("=" * 60)
print(f"""
  Modèle retenu       : {modele_gagnant}
  MAE  Prophet (test) : {mae_test_p:.1f}
  MAE  SARIMA  (test) : {mae_test_s:.1f}

  Fichiers produits :
    volumes_prophet_vs_sarima.png
    previsions_volumes_j30.csv

✅ Bloc 3A terminé.
""")

## BLOC 3B — PRÉVISION DE LA PRODUCTIVITÉ / CHARGE (Ridge vs Elastic Net vs XGBoost)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from xgboost import XGBRegressor

print("=" * 60)
print("BLOC 3B - PRÉVISION DE LA CHARGE PAR ETP (Ridge vs Elastic Net vs XGBoost)")
print("=" * 60)

## CHARGEMENT DES DONNÉES

In [ ]:
print("\n📂 Chargement de dataset_journalier.csv...")

df_jour = pd.read_csv("dataset_journalier.csv", parse_dates=["date"])
df_jour = df_jour.sort_values("date").reset_index(drop=True)

print(f"  ✅ {len(df_jour)} jours | {df_jour['date'].min().date()} → {df_jour['date'].max().date()}")

TARGET = "charge_par_etp"

FEATURES = [
    "volume_lag_1j",  "volume_lag_7j",  "volume_lag_14j", "volume_lag_30j",
    "etp_lag_1j",     "etp_lag_7j",     "etp_lag_14j",    "etp_lag_30j",
    "volume_moy_7j",  "charge_moy_7j",
    "jour_semaine",   "mois",           "trimestre",       "semaine_du_mois",
    "est_fin_de_mois","est_fin_trimestre","est_lundi",     "est_vendredi",
    "etp_disponible", "taux_complexes",
]

df_model = df_jour[FEATURES + [TARGET, "date"]].dropna().copy()
print(f"\n  Lignes utilisables après dropna : {len(df_model)} / {len(df_jour)}")

# ── Split temporel 70% / 15% / 15% ─────────────────────────
n       = len(df_model)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

df_train = df_model.iloc[:n_train]
df_val   = df_model.iloc[n_train : n_train + n_val]
df_test  = df_model.iloc[n_train + n_val :]

print(f"\n  Split chronologique :")
print(f"    Train      : {len(df_train)} jours ({df_train['date'].min().date()} → {df_train['date'].max().date()})")
print(f"    Validation : {len(df_val)} jours ({df_val['date'].min().date()} → {df_val['date'].max().date()})")
print(f"    Test       : {len(df_test)} jours ({df_test['date'].min().date()} → {df_test['date'].max().date()})")

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_val,   y_val   = df_val[FEATURES],   df_val[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]


def metriques(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"\n  📊 Métriques [{label}]")
    print(f"    MAE  : {mae:.2f} dossiers/ETP")
    print(f"    RMSE : {rmse:.2f} dossiers/ETP")
    print(f"    MSE  : {mse:.2f}")
    print(f"    R²   : {r2:.3f}")
    return mae, rmse, mse, r2

## MODÈLE 1 — RIDGE

In [ ]:
print("\n" + "=" * 60)
print("MODÈLE 1 — RIDGE")
print("=" * 60)

# Standardisation nécessaire pour les modèles linéaires régularisés
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

model_ridge = Ridge(alpha=1.0, random_state=42)
model_ridge.fit(X_train_sc, y_train)

pred_val_ridge  = model_ridge.predict(X_val_sc)
pred_test_ridge = model_ridge.predict(X_test_sc)

metriques(y_val, pred_val_ridge, "Ridge - Validation")
mae_ridge, rmse_ridge, mse_ridge, r2_ridge = metriques(y_test, pred_test_ridge, "Ridge - Test")

## MODÈLE 2 — ELASTIC NET

In [ ]:
print("\n" + "=" * 60)
print("MODÈLE 2 — ELASTIC NET")
print("=" * 60)

model_elastic = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=5000)
model_elastic.fit(X_train_sc, y_train)

pred_val_elastic  = model_elastic.predict(X_val_sc)
pred_test_elastic = model_elastic.predict(X_test_sc)

metriques(y_val, pred_val_elastic, "Elastic Net - Validation")
mae_elastic, rmse_elastic, mse_elastic, r2_elastic = metriques(y_test, pred_test_elastic, "Elastic Net - Test")

# Variables éliminées par Elastic Net (coefficient = 0)
coeffs = pd.Series(model_elastic.coef_, index=FEATURES)
elimines = coeffs[coeffs == 0].index.tolist()
print(f"\n  Variables éliminées par Elastic Net (coef=0) : {len(elimines)}")
for v in elimines:
    print(f"    - {v}")

## MODÈLE 3 — XGBOOST (comparatif avancé)

In [ ]:
print("\n" + "=" * 60)
print("MODÈLE 3 — XGBOOST")
print("=" * 60)

model_xgb = XGBRegressor(
    n_estimators          = 300,
    learning_rate         = 0.05,
    max_depth             = 4,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    random_state          = 42,
    early_stopping_rounds = 20,
    eval_metric           = "mae",
    verbosity             = 0
)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"  ✅ Meilleure itération : {model_xgb.best_iteration}")

pred_val_xgb  = model_xgb.predict(X_val)
pred_test_xgb = model_xgb.predict(X_test)

metriques(y_val, pred_val_xgb, "XGBoost - Validation")
mae_xgb, rmse_xgb, mse_xgb, r2_xgb = metriques(y_test, pred_test_xgb, "XGBoost - Test")

## WALK-FORWARD VALIDATION (sur la partie train) — Ridge vs ElasticNet

In [ ]:
print("\n" + "=" * 60)
print("WALK-FORWARD VALIDATION (sur train, fenêtre glissante)")
print("=" * 60)
print("\n  Principe : on ré-entraîne périodiquement sur une fenêtre glissante,")
print("  on prédit le bloc suivant, et on moyenne les erreurs sur tous les blocs.")
print("  Sert ici à vérifier la stabilité de Ridge/ElasticNet (tuning + robustesse).")

def walk_forward(X, y, model_class, model_kwargs, taille_fenetre=120, taille_pred=14):
    maes = []
    i = taille_fenetre
    while i + taille_pred <= len(X):
        X_tr, y_tr = X.iloc[i - taille_fenetre:i], y.iloc[i - taille_fenetre:i]
        X_pr, y_pr = X.iloc[i:i + taille_pred],     y.iloc[i:i + taille_pred]

        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_tr)
        X_pr_sc = sc.transform(X_pr)

        m = model_class(**model_kwargs)
        m.fit(X_tr_sc, y_tr)
        pred = m.predict(X_pr_sc)
        maes.append(mean_absolute_error(y_pr, pred))

        i += taille_pred
    return maes

X_tr_full, y_tr_full = df_train[FEATURES], df_train[TARGET]

maes_ridge_wf   = walk_forward(X_tr_full, y_tr_full, Ridge,      {"alpha": 1.0, "random_state": 42})
maes_elastic_wf = walk_forward(X_tr_full, y_tr_full, ElasticNet, {"alpha": 0.1, "l1_ratio": 0.5, "random_state": 42, "max_iter": 5000})

print(f"\n  Ridge       : MAE walk-forward moyen = {np.mean(maes_ridge_wf):.2f} (sur {len(maes_ridge_wf)} fenêtres)")
print(f"  Elastic Net : MAE walk-forward moyen = {np.mean(maes_elastic_wf):.2f} (sur {len(maes_elastic_wf)} fenêtres)")

## COMPARAISON ET SÉLECTION DU MODÈLE

In [ ]:
print("\n" + "=" * 60)
print("COMPARAISON RIDGE vs ELASTIC NET vs XGBOOST (jeu de test)")
print("=" * 60)

comparaison = pd.DataFrame({
    "Modèle":  ["Ridge", "Elastic Net", "XGBoost"],
    "MAE":     [mae_ridge, mae_elastic, mae_xgb],
    "RMSE":    [rmse_ridge, rmse_elastic, rmse_xgb],
    "MSE":     [mse_ridge, mse_elastic, mse_xgb],
    "R²":      [r2_ridge, r2_elastic, r2_xgb],
})
print("\n" + comparaison.to_string(index=False))

# Choix entre Ridge et Elastic Net (régression de référence)
if mae_elastic <= mae_ridge:
    modele_lineaire_retenu = "Elastic Net"
    mae_lineaire_retenu = mae_elastic
else:
    modele_lineaire_retenu = "Ridge"
    mae_lineaire_retenu = mae_ridge

print(f"\n  ✅ Modèle linéaire retenu (référence) : {modele_lineaire_retenu} (MAE={mae_lineaire_retenu:.2f})")

if mae_xgb < mae_lineaire_retenu:
    gain_pct = (mae_lineaire_retenu - mae_xgb) / mae_lineaire_retenu * 100
    print(f"  ✅ XGBoost apporte un gain de {gain_pct:.1f}% par rapport à {modele_lineaire_retenu} → retenu pour la production")
    modele_final = "XGBoost"
else:
    print(f"  ✅ {modele_lineaire_retenu} reste compétitif face à XGBoost → retenu pour la production (plus simple/interprétable)")
    modele_final = modele_lineaire_retenu

## GRAPHIQUE — PRÉVISION VS RÉEL (TEST)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_test["date"], y_test.values, label="Réel", color="#2c7bb6", linewidth=1.6)
ax.plot(df_test["date"], pred_test_ridge,   label="Ridge",       color="#7570b3", linewidth=1.2, linestyle="--")
ax.plot(df_test["date"], pred_test_elastic, label="Elastic Net", color="#1b9e77", linewidth=1.2, linestyle="-.")
ax.plot(df_test["date"], pred_test_xgb,     label="XGBoost",     color="#d7191c", linewidth=1.4, linestyle=":")
ax.set_title("Charge par ETP — Ridge vs Elastic Net vs XGBoost (jeu de test)",
              fontsize=12, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Charge (dossiers/ETP)")
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig("charge_ridge_vs_elastic_vs_xgb.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  ✅ Graphique sauvegardé : charge_ridge_vs_elastic_vs_xgb.png")

## EXPORT DU MODÈLE RETENU

In [ ]:
import joblib

if modele_final == "XGBoost":
    joblib.dump({
        "model": model_xgb, "features": FEATURES, "scaler": None,
        "mae_test": mae_xgb, "rmse_test": rmse_xgb, "r2_test": r2_xgb,
        "type": "xgboost",
    }, "model_charge_etp.pkl")
elif modele_final == "Elastic Net":
    joblib.dump({
        "model": model_elastic, "features": FEATURES, "scaler": scaler,
        "mae_test": mae_elastic, "rmse_test": rmse_elastic, "r2_test": r2_elastic,
        "type": "elastic_net",
    }, "model_charge_etp.pkl")
else:
    joblib.dump({
        "model": model_ridge, "features": FEATURES, "scaler": scaler,
        "mae_test": mae_ridge, "rmse_test": rmse_ridge, "r2_test": r2_ridge,
        "type": "ridge",
    }, "model_charge_etp.pkl")

print(f"  ✅ model_charge_etp.pkl sauvegardé (modèle retenu : {modele_final})")

## RAPPORT FINAL

In [ ]:
print("\n" + "=" * 60)
print("RAPPORT FINAL - BLOC 3B")
print("=" * 60)
print(f"""
  Modèle retenu pour la production : {modele_final}

  Comparatif (test) :
    Ridge       -> MAE={mae_ridge:.2f}  RMSE={rmse_ridge:.2f}  R²={r2_ridge:.3f}
    Elastic Net -> MAE={mae_elastic:.2f}  RMSE={rmse_elastic:.2f}  R²={r2_elastic:.3f}
    XGBoost     -> MAE={mae_xgb:.2f}  RMSE={rmse_xgb:.2f}  R²={r2_xgb:.3f}

  Walk-forward (train, fenêtre 120j / pas 14j) :
    Ridge       -> MAE moyen = {np.mean(maes_ridge_wf):.2f}
    Elastic Net -> MAE moyen = {np.mean(maes_elastic_wf):.2f}

  Fichiers produits :
    charge_ridge_vs_elastic_vs_xgb.png
    model_charge_etp.pkl

✅ Bloc 3B terminé.
""")

## BLOC 3C — CLASSIFICATION DU RISQUE DE RETARD (RandomForest vs Decision Tree)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

print("=" * 60)
print("BLOC 3C - CLASSIFICATION DU RISQUE DE RETARD (RandomForest vs Decision Tree)")
print("=" * 60)

## CHARGEMENT DES DONNÉES

In [ ]:
print("\n📂 Chargement de dataset_features.csv...")

df = pd.read_csv("dataset_features.csv", parse_dates=[
    "Date_Creation", "Date_Prise_En_Charge", "Date_Cloture"
])
print(f"  ✅ {len(df)} lignes | {df.shape[1]} colonnes")

## CONSTRUCTION DE LA CIBLE : EST_EN_RETARD

In [ ]:
print("\n🎯 Construction de la cible 'est_en_retard'...")

# Seuil de retard = P75 de la durée de traitement réelle (cohérent avec
# le seuil de surcharge déjà utilisé pour charge_par_etp en feature engineering)
seuil_retard_j = df["duree_traitement_reelle_j"].quantile(0.75)
print(f"  Seuil de retard : durée > {seuil_retard_j:.1f} jours (P75)")

df["est_en_retard"] = (df["duree_traitement_reelle_j"] > seuil_retard_j).astype(int)

# On ne garde que les tâches clôturées avec une durée connue
df_clf = df[df["duree_traitement_reelle_j"].notna()].copy()
print(f"  Lignes avec durée connue : {len(df_clf)} / {len(df)}")
print(f"  Répartition cible : {df_clf['est_en_retard'].value_counts().to_dict()}")
print(f"  Taux de retard : {df_clf['est_en_retard'].mean()*100:.1f}%")

## FEATURES

In [ ]:
FEATURES_NUM = [
    "complexite_num", "coeff_productivite_contrat",
    "charge_journaliere_nb_taches", "etp_agent_jour",
    "etp_disponible_service_jour", "charge_par_etp_service",
    "anciennete_agent_mois", "delai_prise_en_charge_j",
    "jour_semaine", "mois", "trimestre", "semaine_du_mois",
    "est_fin_de_mois", "est_fin_trimestre", "est_lundi", "est_vendredi",
    "est_absent", "alerte_surcharge_service",
]

# Encodage one-hot du Service et du Type_Contrat (variables catégorielles)
df_clf = pd.get_dummies(df_clf, columns=["Service", "Type_Contrat"], prefix=["svc", "ctr"])
features_dummies = [c for c in df_clf.columns if c.startswith("svc_") or c.startswith("ctr_")]

FEATURES = FEATURES_NUM + features_dummies
FEATURES = [f for f in FEATURES if f in df_clf.columns]

TARGET = "est_en_retard"

df_model = df_clf[FEATURES + [TARGET, "Date_Creation"]].dropna().copy()
df_model = df_model.sort_values("Date_Creation").reset_index(drop=True)
print(f"\n  Lignes utilisables après dropna : {len(df_model)} / {len(df_clf)}")
print(f"  Nombre de features : {len(FEATURES)}")

## SPLIT TEMPOREL 70 / 15 / 15

In [ ]:
n       = len(df_model)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

df_train = df_model.iloc[:n_train]
df_val   = df_model.iloc[n_train : n_train + n_val]
df_test  = df_model.iloc[n_train + n_val :]

print(f"\n  Split chronologique :")
print(f"    Train      : {len(df_train)} lignes ({df_train['Date_Creation'].min().date()} → {df_train['Date_Creation'].max().date()})")
print(f"    Validation : {len(df_val)} lignes ({df_val['Date_Creation'].min().date()} → {df_val['Date_Creation'].max().date()})")
print(f"    Test       : {len(df_test)} lignes ({df_test['Date_Creation'].min().date()} → {df_test['Date_Creation'].max().date()})")

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_val,   y_val   = df_val[FEATURES],   df_val[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]


def evaluer(model, X, y, label):
    pred = model.predict(X)
    acc  = accuracy_score(y, pred)
    print(f"\n  📊 [{label}]")
    print(f"    Accuracy : {acc:.3f}")
    print(classification_report(y, pred, target_names=["Normal", "En retard"], zero_division=0))
    return acc, pred

## MODÈLE 1 — DECISION TREE

In [ ]:
print("\n" + "=" * 60)
print("MODÈLE 1 — DECISION TREE")
print("=" * 60)

model_tree = DecisionTreeClassifier(
    max_depth=6, min_samples_leaf=50, class_weight="balanced", random_state=42
)
model_tree.fit(X_train, y_train)

evaluer(model_tree, X_val, y_val, "Decision Tree - Validation")
acc_tree, pred_tree_test = evaluer(model_tree, X_test, y_test, "Decision Tree - Test")

## MODÈLE 2 — RANDOM FOREST

In [ ]:
print("\n" + "=" * 60)
print("MODÈLE 2 — RANDOM FOREST")
print("=" * 60)

model_rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=20,
    class_weight="balanced", random_state=42, n_jobs=-1
)
model_rf.fit(X_train, y_train)

evaluer(model_rf, X_val, y_val, "Random Forest - Validation")
acc_rf, pred_rf_test = evaluer(model_rf, X_test, y_test, "Random Forest - Test")

## WALK-FORWARD VALIDATION (sur train, fenêtre glissante)

In [ ]:
print("\n" + "=" * 60)
print("WALK-FORWARD VALIDATION (sur train, fenêtre glissante)")
print("=" * 60)

def walk_forward_clf(X, y, model_class, model_kwargs, taille_fenetre=50000, taille_pred=10000):
    accs = []
    i = taille_fenetre
    while i + taille_pred <= len(X):
        X_tr, y_tr = X.iloc[i - taille_fenetre:i], y.iloc[i - taille_fenetre:i]
        X_pr, y_pr = X.iloc[i:i + taille_pred],     y.iloc[i:i + taille_pred]
        m = model_class(**model_kwargs)
        m.fit(X_tr, y_tr)
        accs.append(accuracy_score(y_pr, m.predict(X_pr)))
        i += taille_pred
    return accs

accs_tree_wf = walk_forward_clf(
    df_train[FEATURES], df_train[TARGET], DecisionTreeClassifier,
    {"max_depth": 6, "min_samples_leaf": 50, "class_weight": "balanced", "random_state": 42}
)
accs_rf_wf = walk_forward_clf(
    df_train[FEATURES], df_train[TARGET], RandomForestClassifier,
    {"n_estimators": 100, "max_depth": 10, "min_samples_leaf": 20,
     "class_weight": "balanced", "random_state": 42, "n_jobs": -1}
)

print(f"\n  Decision Tree : Accuracy walk-forward moyenne = {np.mean(accs_tree_wf):.3f} (sur {len(accs_tree_wf)} fenêtres)")
print(f"  Random Forest : Accuracy walk-forward moyenne = {np.mean(accs_rf_wf):.3f} (sur {len(accs_rf_wf)} fenêtres)")

## COMPARAISON ET SÉLECTION DU MODÈLE

In [ ]:
print("\n" + "=" * 60)
print("COMPARAISON DECISION TREE vs RANDOM FOREST (jeu de test)")
print("=" * 60)

comparaison = pd.DataFrame({
    "Modèle": ["Decision Tree", "Random Forest"],
    "Accuracy (test)": [acc_tree, acc_rf],
    "Accuracy (walk-forward)": [np.mean(accs_tree_wf), np.mean(accs_rf_wf)],
})
print("\n" + comparaison.to_string(index=False))

if acc_rf >= acc_tree:
    modele_final = "Random Forest"
    model_final = model_rf
    acc_final = acc_rf
else:
    modele_final = "Decision Tree"
    model_final = model_tree
    acc_final = acc_tree

print(f"\n  ✅ Modèle retenu pour la production : {modele_final} (Accuracy={acc_final:.3f})")

## GRAPHIQUE — MATRICE DE CONFUSION + IMPORTANCE DES VARIABLES

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, model_final.predict(X_test))
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Normal", "En retard"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Normal", "En retard"])
axes[0].set_xlabel("Prédit"); axes[0].set_ylabel("Réel")
axes[0].set_title(f"Matrice de confusion — {modele_final}")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                      color="white" if cm[i, j] > cm.max()/2 else "black")

importance = pd.Series(model_final.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(12)
importance.plot(kind="barh", ax=axes[1], color="#2c7bb6")
axes[1].set_title(f"Importance des variables — {modele_final} (top 12)")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("classification_retard_resultats.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  ✅ Graphique sauvegardé : classification_retard_resultats.png")

## EXPORT DU MODÈLE RETENU

In [ ]:
joblib.dump({
    "model": model_final,
    "features": FEATURES,
    "seuil_retard_j": seuil_retard_j,
    "accuracy_test": acc_final,
    "type": "random_forest" if modele_final == "Random Forest" else "decision_tree",
}, "model_classification_retard.pkl")
print("  ✅ model_classification_retard.pkl sauvegardé")

## RAPPORT FINAL

In [ ]:
print("\n" + "=" * 60)
print("RAPPORT FINAL - BLOC 3C")
print("=" * 60)
print(f"""
  Cible : est_en_retard = 1 si durée traitement > {seuil_retard_j:.1f} jours (P75)
  Taux de retard dans les données : {df_clf['est_en_retard'].mean()*100:.1f}%

  Modèle retenu pour la production : {modele_final}

  Comparatif (test) :
    Decision Tree -> Accuracy = {acc_tree:.3f}
    Random Forest -> Accuracy = {acc_rf:.3f}

  Walk-forward (train) :
    Decision Tree -> Accuracy moyenne = {np.mean(accs_tree_wf):.3f}
    Random Forest -> Accuracy moyenne = {np.mean(accs_rf_wf):.3f}

  Fichiers produits :
    classification_retard_resultats.png
    model_classification_retard.pkl

✅ Bloc 3C terminé.
""")

## BLOC 3D — DÉTECTION D'ANOMALIES (Isolation Forest + SHAP)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import shap
import joblib

print("=" * 60)
print("BLOC 3D - DÉTECTION D'ANOMALIES (Isolation Forest + SHAP)")
print("=" * 60)

## CHARGEMENT DES DONNÉES

In [ ]:
print("\n📂 Chargement de dataset_features.csv...")

df = pd.read_csv("dataset_features.csv", parse_dates=["Date_Creation"])
print(f"  ✅ {len(df)} lignes | {df.shape[1]} colonnes")

## FEATURES POUR LA DÉTECTION D'ANOMALIES

In [ ]:
# On se concentre sur des indicateurs de cohérence individuelle de la tâche :
# une anomalie = une déclaration incohérente (temps/volume/productivité hors norme),
# pas une "mauvaise performance" — signal pour le manager, jamais décision RH automatique.
FEATURES = [
    "Temps_Passe_Declare_Min",
    "Volume_Dossiers",
    "productivite_dossiers_par_heure",
    "duree_traitement_reelle_j",
    "delai_prise_en_charge_j",
    "complexite_num",
    "coeff_productivite_contrat",
]

df_model = df[FEATURES + ["ID_Tache", "Matricule_Agent", "Service", "Date_Creation",
                           "Type_Processus", "Complexite"]].dropna(subset=FEATURES).copy()
print(f"\n  Lignes utilisables après dropna : {len(df_model)} / {len(df)}")

## SPLIT TEMPOREL 70 / 15 / 15

In [ ]:
df_model = df_model.sort_values("Date_Creation").reset_index(drop=True)
n       = len(df_model)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

df_train = df_model.iloc[:n_train]
df_val   = df_model.iloc[n_train : n_train + n_val]
df_test  = df_model.iloc[n_train + n_val :]

print(f"\n  Split chronologique :")
print(f"    Train      : {len(df_train)} lignes")
print(f"    Validation : {len(df_val)} lignes")
print(f"    Test       : {len(df_test)} lignes")

scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[FEATURES])
X_val   = scaler.transform(df_val[FEATURES])
X_test  = scaler.transform(df_test[FEATURES])

## ENTRAÎNEMENT — ISOLATION FOREST

In [ ]:
print("\n" + "=" * 60)
print("ISOLATION FOREST")
print("=" * 60)

# contamination = proportion attendue d'anomalies (~2%, ordre de grandeur
# réaliste pour des erreurs de déclaration ponctuelles)
model_if = IsolationForest(
    n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1
)
model_if.fit(X_train)

print("\n  Entraînement terminé sur le train set.")

## INJECTION D'ANOMALIES CONNUES DANS LE TEST (validation)

In [ ]:
print("\n🧪 Injection d'anomalies synthétiques dans le test set...")

# On injecte des valeurs extrêmes connues (productivité aberrante,
# temps négatif/nul incohérent avec le volume) pour vérifier que le
# modèle les détecte effectivement — méthode recommandée quand on n'a
# pas de labels d'anomalie réels.
df_test_aug = df_test.copy().reset_index(drop=True)
n_inject = max(10, int(len(df_test_aug) * 0.01))
idx_inject = np.random.RandomState(42).choice(len(df_test_aug), size=n_inject, replace=False)

df_test_aug["anomalie_injectee"] = 0
for idx in idx_inject:
    # Productivité extrême : volume énorme pour un temps très court
    df_test_aug.loc[idx, "Volume_Dossiers"] = df_test_aug.loc[idx, "Volume_Dossiers"] * 50
    df_test_aug.loc[idx, "productivite_dossiers_par_heure"] = (
        df_test_aug.loc[idx, "Volume_Dossiers"] / (df_test_aug.loc[idx, "Temps_Passe_Declare_Min"] / 60)
    )
    df_test_aug.loc[idx, "anomalie_injectee"] = 1

print(f"  ✅ {n_inject} anomalies synthétiques injectées (productivité x50)")

X_test_aug = scaler.transform(df_test_aug[FEATURES])
pred_test_aug = model_if.predict(X_test_aug)  # -1 = anomalie, 1 = normal
df_test_aug["anomalie_detectee"] = (pred_test_aug == -1).astype(int)

taux_detection = df_test_aug.loc[df_test_aug["anomalie_injectee"] == 1, "anomalie_detectee"].mean()
print(f"  ✅ Taux de détection des anomalies injectées : {taux_detection*100:.1f}%")

## SCORES D'ANOMALIE SUR LE DATASET COMPLET

In [ ]:
print("\n📊 Calcul des scores d'anomalie sur l'ensemble du dataset...")

X_full = scaler.transform(df_model[FEATURES])
df_model["score_anomalie"] = model_if.decision_function(X_full)  # plus bas = plus anormal
df_model["est_anomalie"] = (model_if.predict(X_full) == -1).astype(int)

print(f"  Taux d'anomalies détectées : {df_model['est_anomalie'].mean()*100:.2f}%")
print(f"\n  Répartition des anomalies par service :")
print(df_model.groupby("Service")["est_anomalie"].mean().sort_values(ascending=False) * 100)

## SHAP — EXPLICABILITÉ DES ANOMALIES

In [ ]:
print("\n" + "=" * 60)
print("SHAP — EXPLICABILITÉ DES ANOMALIES")
print("=" * 60)

# Échantillon pour SHAP (TreeExplainer fonctionne avec IsolationForest
# via le wrapper générique, mais coûteux sur 700k+ lignes -> échantillon)
echantillon = df_model.sample(n=min(2000, len(df_model)), random_state=42)
X_echantillon = scaler.transform(echantillon[FEATURES])

print("\n  Calcul des valeurs SHAP sur un échantillon de 2000 lignes...")
explainer = shap.TreeExplainer(model_if)
shap_values = explainer.shap_values(X_echantillon)

# Importance moyenne (valeur absolue) par feature
shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=FEATURES
).sort_values(ascending=False)
print("\n  Importance moyenne des features (SHAP) :")
print(shap_importance.to_string())

# Graphique summary plot
shap.summary_plot(shap_values, X_echantillon, feature_names=FEATURES, show=False)
plt.tight_layout()
plt.savefig("shap_anomalies_summary.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  ✅ Graphique sauvegardé : shap_anomalies_summary.png")

## EXPLICATION DÉTAILLÉE DES TOP ANOMALIES (pour le dashboard)

In [ ]:
print("\n📋 Top 10 anomalies les plus marquées (dataset complet)...")

top_anomalies = df_model.sort_values("score_anomalie").head(10).copy()
print(top_anomalies[["ID_Tache", "Matricule_Agent", "Service", "Type_Processus",
                      "Complexite", "Temps_Passe_Declare_Min", "Volume_Dossiers",
                      "productivite_dossiers_par_heure", "score_anomalie"]].to_string(index=False))

# Export pour le dashboard
colonnes_export = ["ID_Tache", "Matricule_Agent", "Service", "Date_Creation",
                    "Type_Processus", "Complexite", "Temps_Passe_Declare_Min",
                    "Volume_Dossiers", "productivite_dossiers_par_heure",
                    "score_anomalie", "est_anomalie"]
df_model[colonnes_export].to_csv("anomalies_detectees.csv", index=False, encoding="utf-8")
print("\n  ✅ anomalies_detectees.csv exporté")

## EXPORT DU MODÈLE

In [ ]:
joblib.dump({
    "model": model_if,
    "scaler": scaler,
    "features": FEATURES,
    "taux_detection_test": taux_detection,
    "taux_anomalies_global": df_model["est_anomalie"].mean(),
}, "model_isolation_forest.pkl")
print("  ✅ model_isolation_forest.pkl sauvegardé")

## RAPPORT FINAL

In [ ]:
print("\n" + "=" * 60)
print("RAPPORT FINAL - BLOC 3D")
print("=" * 60)
print(f"""
  Taux de contamination configuré : 2%
  Taux d'anomalies détectées (global) : {df_model['est_anomalie'].mean()*100:.2f}%

  Validation par injection :
    {n_inject} anomalies synthétiques injectées (productivité x50)
    Taux de détection : {taux_detection*100:.1f}%

  Top feature explicative (SHAP) : {shap_importance.index[0]}

  Fichiers produits :
    shap_anomalies_summary.png
    anomalies_detectees.csv
    model_isolation_forest.pkl

✅ Bloc 3D terminé.
""")